<a href="https://colab.research.google.com/github/cn8972/Echo-Bot/blob/main/End-to-End%20Cognitive%20Offloading%20Prediction%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Cognitive Offloading Predictor - Fixed Version
# Google Colab ready
# ============================================

# 1. Install required packages
!pip -q install gradio joblib scikit-learn pandas numpy matplotlib

# 2. Imports
import os
import io
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

import gradio as gr

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 3. Configuration
REGRESSION_TARGET = "Cognitive_Offloading_Index"
CLASSIFICATION_TARGET = "Cognitive_Offloading_Level"
MODEL_BUNDLE_PATH = "/content/cognitive_offloading_bundle.joblib"

# 4. Helper functions
def validate_dataframe(df: pd.DataFrame):
    if REGRESSION_TARGET not in df.columns:
        raise ValueError(
            f"The dataset must include '{REGRESSION_TARGET}' as the regression target."
        )
    if df.shape[0] < 30:
        raise ValueError("The dataset is too small. Use at least 30 records.")
    return True


def create_offloading_class(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create Low / Medium / High classes from the Cognitive Offloading Index
    if the class label does not already exist.
    """
    df = df.copy()
    if CLASSIFICATION_TARGET not in df.columns:
        q1 = df[REGRESSION_TARGET].quantile(0.33)
        q2 = df[REGRESSION_TARGET].quantile(0.66)

        def label_fn(x):
            if x <= q1:
                return "Low"
            elif x <= q2:
                return "Medium"
            return "High"

        df[CLASSIFICATION_TARGET] = df[REGRESSION_TARGET].apply(label_fn)

    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create derived features relevant to cognitive offloading.
    """
    df = df.copy()

    def get_numeric_series(column_name, default_value=0.0):
        if column_name in df.columns:
            return pd.to_numeric(df[column_name], errors="coerce").fillna(default_value)
        return pd.Series([default_value] * len(df), index=df.index, dtype=float)

    verification_rate = get_numeric_series("Verification_Rate")
    performance_drop = get_numeric_series("Performance_Drop_Without_AI")
    trust_score = get_numeric_series("Trust_in_AI_Score")
    ai_use_frequency = get_numeric_series("AI_Use_Frequency")
    ai_acceptance = get_numeric_series("AI_Output_Acceptance_Rate")
    edit_distance = get_numeric_series("AI_Output_Edit_Distance")
    mental_workload = get_numeric_series("Perceived_Mental_Workload")
    cognitive_fatigue = get_numeric_series("Cognitive_Fatigue_Level")
    interruptions = get_numeric_series("Interruptions_Count")

    df["Reliance_Score"] = (ai_use_frequency + ai_acceptance) / 2
    df["Oversight_Score"] = (
        verification_rate + (1 - edit_distance.clip(lower=0, upper=1))
    ) / 2
    df["Dependency_Score"] = performance_drop
    df["Cognitive_Strain_Score"] = (
        mental_workload + cognitive_fatigue + interruptions
    ) / 3

    verification_scaled = (
        verification_rate * 100 if verification_rate.max() <= 1 else verification_rate
    )
    df["Trust_Verification_Gap"] = trust_score - verification_scaled

    return df


def split_features(df: pd.DataFrame):
    excluded = [REGRESSION_TARGET, CLASSIFICATION_TARGET]
    X = df.drop(columns=[c for c in excluded if c in df.columns])
    y_reg = df[REGRESSION_TARGET]
    y_cls = df[CLASSIFICATION_TARGET]
    return X, y_reg, y_cls


def build_preprocessor(X: pd.DataFrame):
    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    return preprocessor, numeric_cols, categorical_cols


def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = list(numeric_cols)

    if categorical_cols:
        ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        encoded_names = ohe.get_feature_names_out(categorical_cols).tolist()
        feature_names.extend(encoded_names)

    return feature_names


def get_feature_importance(pipeline, numeric_cols, categorical_cols, top_n=15):
    preprocessor = pipeline.named_steps["preprocessor"]
    model = pipeline.named_steps["model"]

    feature_names = get_feature_names(preprocessor, numeric_cols, categorical_cols)
    importances = model.feature_importances_

    fi = pd.DataFrame(
        {"Feature": feature_names, "Importance": importances}
    ).sort_values("Importance", ascending=False)

    return fi.head(top_n)


def train_models(df: pd.DataFrame):
    validate_dataframe(df)
    df = create_offloading_class(df)
    df = engineer_features(df)

    X, y_reg, y_cls = split_features(df)
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X)

    X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
        X,
        y_reg,
        y_cls,
        test_size=0.2,
        random_state=42,
        stratify=y_cls,
    )

    reg_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=250,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            )),
        ]
    )

    cls_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=250,
                max_depth=10,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )),
        ]
    )

    reg_pipeline.fit(X_train, y_reg_train)
    cls_pipeline.fit(X_train, y_cls_train)

    y_reg_pred = reg_pipeline.predict(X_test)
    y_cls_pred = cls_pipeline.predict(X_test)

    reg_metrics = {
        "MAE": float(mean_absolute_error(y_reg_test, y_reg_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))),
        "R2": float(r2_score(y_reg_test, y_reg_pred)),
    }

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_cls_test,
        y_cls_pred,
        average="weighted",
        zero_division=0,
    )

    cls_metrics = {
        "Accuracy": float(accuracy_score(y_cls_test, y_cls_pred)),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "Classification_Report": classification_report(
            y_cls_test, y_cls_pred, zero_division=0
        ),
    }

    reg_feature_importance = get_feature_importance(
        reg_pipeline, numeric_cols, categorical_cols
    )
    cls_feature_importance = get_feature_importance(
        cls_pipeline, numeric_cols, categorical_cols
    )

    bundle = {
        "reg_pipeline": reg_pipeline,
        "cls_pipeline": cls_pipeline,
        "reg_metrics": reg_metrics,
        "cls_metrics": cls_metrics,
        "reg_feature_importance": reg_feature_importance,
        "cls_feature_importance": cls_feature_importance,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "feature_columns": X.columns.tolist(),
    }

    joblib.dump(bundle, MODEL_BUNDLE_PATH)
    return bundle, df


# 5. Upload and load dataset
def load_dataset():
    if IN_COLAB:
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        filename = next(iter(uploaded))
        if filename.endswith(".csv"):
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
        elif filename.endswith(".xlsx"):
            df = pd.read_excel(io.BytesIO(uploaded[filename]))
        else:
            raise ValueError("Upload a CSV or XLSX file.")
    else:
        raise RuntimeError("This upload helper is intended for Google Colab.")
    return df


# 6. Utility for scenario prediction
def build_input_row(input_dict, feature_columns):
    row = {}
    for col in feature_columns:
        row[col] = input_dict.get(col, np.nan)
    return pd.DataFrame([row])


# 7. Train models
print("Upload your dataset to begin training.")
df = load_dataset()
bundle, trained_df = train_models(df)

print("\nRegression Metrics")
print(json.dumps(bundle["reg_metrics"], indent=2))

print("\nClassification Metrics")
print(json.dumps(
    {k: v for k, v in bundle["cls_metrics"].items() if k != "Classification_Report"},
    indent=2
))
print("\nClassification Report")
print(bundle["cls_metrics"]["Classification_Report"])

print("\nTop Regression Features")
display(bundle["reg_feature_importance"])

print("\nTop Classification Features")
display(bundle["cls_feature_importance"])


# 8. Gradio app functions
feature_columns = bundle["feature_columns"]

def dataset_overview():
    summary = {
        "rows": int(trained_df.shape[0]),
        "columns": int(trained_df.shape[1]),
        "regression_target_mean": float(trained_df[REGRESSION_TARGET].mean()),
        "class_distribution": trained_df[CLASSIFICATION_TARGET].value_counts().to_dict(),
    }
    return json.dumps(summary, indent=2)


def model_diagnostics():
    payload = {
        "regression_metrics": bundle["reg_metrics"],
        "classification_metrics": {
            k: v for k, v in bundle["cls_metrics"].items()
            if k != "Classification_Report"
        },
        "classification_report": bundle["cls_metrics"]["Classification_Report"],
    }
    return json.dumps(payload, indent=2)


def predict_scenario(json_input):
    try:
        input_dict = json.loads(json_input)
        X_new = build_input_row(input_dict, feature_columns)
        X_new = engineer_features(X_new)

        reg_pred = bundle["reg_pipeline"].predict(X_new)[0]
        cls_pred = bundle["cls_pipeline"].predict(X_new)[0]

        result = {
            "Predicted_Cognitive_Offloading_Index": float(reg_pred),
            "Predicted_Cognitive_Offloading_Level": str(cls_pred),
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Prediction error: {str(e)}"


def feature_importance_text():
    reg_text = bundle["reg_feature_importance"].to_string(index=False)
    cls_text = bundle["cls_feature_importance"].to_string(index=False)
    return f"Top Regression Features\n\n{reg_text}\n\nTop Classification Features\n\n{cls_text}"


# 9. Launch Gradio app
with gr.Blocks(title="Cognitive Offloading Predictor") as demo:
    gr.Markdown("# Cognitive Offloading Predictor")
    gr.Markdown(
        "Explore the trained model, inspect diagnostics, and test hypothetical scenarios."
    )

    with gr.Tab("Dataset Overview"):
        overview_btn = gr.Button("Show Dataset Overview")
        overview_output = gr.Textbox(lines=12, label="Overview")
        overview_btn.click(fn=dataset_overview, outputs=overview_output)

    with gr.Tab("Model Diagnostics"):
        diag_btn = gr.Button("Show Diagnostics")
        diag_output = gr.Textbox(lines=20, label="Diagnostics")
        diag_btn.click(fn=model_diagnostics, outputs=diag_output)

    with gr.Tab("Feature Importance"):
        fi_btn = gr.Button("Show Feature Importance")
        fi_output = gr.Textbox(lines=25, label="Feature Importance")
        fi_btn.click(fn=feature_importance_text, outputs=fi_output)

    with gr.Tab("Scenario Testing"):
        example_json = {
            "Occupation": "Software Developer",
            "AI_Use_Frequency": 0.8,
            "Verification_Rate": 0.4,
            "Performance_Drop_Without_AI": 0.35,
            "Trust_in_AI_Score": 62,
            "AI_Output_Acceptance_Rate": 0.75,
            "AI_Output_Edit_Distance": 0.25,
            "Perceived_Mental_Workload": 55,
            "Cognitive_Fatigue_Level": 48,
            "Interruptions_Count": 4
        }

        scenario_input = gr.Textbox(
            lines=14,
            label="Enter scenario as JSON",
            value=json.dumps(example_json, indent=2),
        )
        predict_btn = gr.Button("Predict")
        predict_output = gr.Textbox(lines=8, label="Prediction")
        predict_btn.click(fn=predict_scenario, inputs=scenario_input, outputs=predict_output)

demo.launch(debug=True)

Upload your dataset to begin training.


Saving synthetic_cognitive_offloading_top25_ai_occupations_1500_records -Dataset v2.xlsx to synthetic_cognitive_offloading_top25_ai_occupations_1500_records -Dataset v2.xlsx

Regression Metrics
{
  "MAE": 1.0747202807306913,
  "RMSE": 1.3764354115160238,
  "R2": 0.9945654297980981
}

Classification Metrics
{
  "Accuracy": 1.0,
  "Precision": 1.0,
  "Recall": 1.0,
  "F1": 1.0
}

Classification Report
              precision    recall  f1-score   support

        High       1.00      1.00      1.00       113
         Low       1.00      1.00      1.00        77
      Medium       1.00      1.00      1.00       110

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Top Regression Features


,Feature,Importance
1,AI_Use_Frequency,0.161843
2,Verification_Rate,0.146247
17,Reliance_Score,0.142300
18,Oversight_Score,0.140672
21,Trust_Verification_Gap,0.131847
7,ai_output_acceptance_rate,0.093629
4293,offloading_type_maladaptive_dependency,0.057028
4292,offloading_type_adaptive_support,0.036464
19,Dependency_Score,0.036336
3,Performance_Drop_Without_AI,0.027532



Top Classification Features


,Feature,Importance
3,Performance_Drop_Without_AI,0.109590
18,Oversight_Score,0.085775
19,Dependency_Score,0.085434
2,Verification_Rate,0.078603
4294,offloading_type_mixed,0.077712
21,Trust_Verification_Gap,0.064767
4293,offloading_type_maladaptive_dependency,0.062158
4292,offloading_type_adaptive_support,0.045254
7,ai_output_acceptance_rate,0.031035
8,ai_output_edit_distance,0.024990


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://30804b205a410876ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
